# Rural Road Extraction - Kaggle Pipeline Runner

This notebook downloads the dataset from Google Drive, extracts it, and runs the `MobileViT v2` training pipeline using AMP and dynamic topology-aware clDice loss.

In [ ]:
!pip install gdown -q

# Download the dataset from Google Drive
import gdown
import os

file_id = '1OI0XJ1-ejxd0JS45hBzJbAYe9_2hJJwN'
url = f'https://drive.google.com/uc?id={file_id}'
output = '/kaggle/working/archive.zip'

if not os.path.exists(output):
    print("Downloading dataset...")
    gdown.download(url, output, quiet=False)
else:
    print("Dataset already downloaded.")

if not os.path.exists('/kaggle/working/dataset'):
    print("Extracting dataset...")
    !unzip -q /kaggle/working/archive.zip -d /kaggle/working/dataset
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

In [ ]:
import os
import sys

# Add the project root to Python path so we can import 'src'
# Assuming this notebook is run from the project root or backend directory
if os.path.exists('backend'):
    sys.path.append('backend')
elif os.path.exists('src'):
    sys.path.append('.')

# Kaggle Paths Setup (Adjust these if the extracted zip structure is different!)
# ---------------------------------------------------------
TRAIN_IMG_DIR = '/kaggle/working/dataset/train'
TRAIN_MASK_DIR = '/kaggle/working/dataset/train' # Update if your generated labels are elsewhere

# Validation split
VAL_IMG_DIR = '/kaggle/working/dataset/valid'
VAL_MASK_DIR = '/kaggle/working/dataset/valid'

OUTPUT_DIR = '/kaggle/working/models'

# Path validation
print("--- Validating Paths ---")
for name, path in [('Train Images', TRAIN_IMG_DIR), ('Train Masks', TRAIN_MASK_DIR)]:
    if not os.path.exists(path):
        print(f"\u26A0\uFE0F Warning: Path not found for {name}: {path}\n" 
              f"Please check the extracted folder structure in /kaggle/working/dataset.")
    else:
        print(f"\u2705 {name} path exists: {path}")


## Execute Training Loop


In [ ]:
# Set W&B API Key (Uncomment and replace if logging to Weights & Biases)
# os.environ["WANDB_API_KEY"] = "your_wandb_api_key_here"

# Configuration
EPOCHS = 50
BATCH_SIZE = 16
LR = 1e-3
import os
NUM_WORKERS = min(4, os.cpu_count() or 2)
RUN_NAME = "kaggle-mobilevit-v2-run1"

# Run the MLOps pipeline script
!python backend/scripts/train.py \
    --train_image_dir {TRAIN_IMG_DIR} \
    --train_mask_dir {TRAIN_MASK_DIR} \
    --val_image_dir {VAL_IMG_DIR} \
    --val_mask_dir {VAL_MASK_DIR} \
    --output_dir {OUTPUT_DIR} \
    --run_name {RUN_NAME} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --num_workers {NUM_WORKERS}

print("\n\u2728 Training completed! Best model saved to:", os.path.join(OUTPUT_DIR, "best_model.pth"))